# Eval Overview - Perturbation Retrieval

Our perturbation retrieval evaluation category focuses on how well our model predicts the perturbation impact compared to other similar perturbations we've trained on. This eval recalculates expression deltas per perturbation using the [linear expression decoder](explainer_eval_decoders_v1_0.ipynb). The goal of this evaluation is to see if our model can, for a real gene expression delta, figure out which perturbation would produce it and how many perturbations, on average, we have to review to find it. Given the number of genes, this eval is dominated by predicting minor movements in the housekeeping genes, but an alternative, if we wanted to get more focused, would be to use the top K differentially expressed genes.

For this eval, we first take our perturbations and split them up by modality: DNA, chemical, and target-only. Once we have them split, we go through each perturbation, calculate the control cell's predicted change for every other perturbation in the category, and then see how close the real expression delta is to the predicted changes. We then see where the actual perturbation lands based on rank and take a category average.

In [1]:
import numpy as np

In [2]:
SEED = 1337
np.random.seed(SEED)

## Data Prep

We'll start by preparing our data. For this evaluation, we need the average real expression delta and control expression by perturbation. A "perturbation" is a unique combination of a sequence, target, modality, and mode applied to a cell type, and it can span datasets if appropriate. We'll start by mocking up 9 samples across 7 perturbations. We'll make sure we have 2 samples that simulate target-only perturbations, 2 chemical samples, and different types of DNA-based perturbations for the remaining samples. We'll first calculate the case and control expression, then calculate the delta and aggregate by perturbation. Since we are not predicting expression, at this stage no inference has been made by the BioJEPA-AC model.

In [3]:
num_genes = 8
num_cells = 9

**Perturbations**

We'll first start with our perturbations. Even though we have 9 cells, our staged data will only have 7 unique perturbations. To define a unique perturbation, it's not just about what we target, but the context of it. Because of this, we represent a unique perturbation as $\text{(seq id, targ id, modality id, mode id, cell type)}$. IDs are used since our model keeps the perturbation information in separate caches from our sample expression counts to avoid heavy duplication of information. For this evaluation, we'll add in cells to represent an additional chemical inhibitor and two target-only perturbations so we can show the three retrieval categories.

We'll also create a mapping of the sample to the perturbation showing how multiple cells can share a perturbation. We'll then create a dictionary that maps the modalities to our three different classes of perturbations: DNA-based, chemical-based, and target-only.

In [4]:
pert_keys = [
    (0, 0, 0, 0, 0),   # pert 0: DNA CRISPRi
    (1, 1, 0, 0, 0),   # pert 1: DNA CRISPRi
    (2, 2, 0, 1, 0),   # pert 2: DNA CRISPRa
    (3, 3, 2, 4, 0),   # pert 3: chemical inhibitor
    (4, 5, 2, 4, 0),   # pert 4: chemical inhibitor
    (-1, 4, 0, 2, 0),  # pert 5: target-only, overexpression
    (-1, 6, 0, 3, 0),  # pert 6: target-only, knockout
]

sample_to_pert = [0, 0, 1, 1, 2, 3, 4, 5, 6]

unique_perts = len(pert_keys)

In [5]:
modality_to_pertidx = {'dna': [], 'chemical': [], 'target_only': []}
modality_to_pertidx

{'dna': [], 'chemical': [], 'target_only': []}

In [6]:
for i in range(unique_perts):
    key = pert_keys[i]
    if key[0] < 0 and key[1] >= 0:
        modality_to_pertidx['target_only'].append(i)
    elif key[2] == 0:
        modality_to_pertidx['dna'].append(i)
    elif key[2] == 2:
        modality_to_pertidx['chemical'].append(i)
    else:
        print(f'error classifying pert {i}: {key}')
modality_to_pertidx

{'dna': [0, 1, 2], 'chemical': [3, 4], 'target_only': [5, 6]}

**Control Cells**

Next we'll show the control cell values. Recall that for our inference and ACPredictor training, we pair together a perturbed cell with a random control cell from the same batch. This allows us to calculate an average change in expression per perturbation.

*Note that we're just showing real expression counts here and no inference has been simulated yet.*

In [7]:
real_control = np.array([
    [2.1, 3.4, 1.2, 4.1, 2.6, 3.1, 1.4, 4.6],
    [1.9, 3.6, 0.8, 3.9, 2.4, 2.9, 1.6, 4.4],
    [2.0, 3.3, 1.1, 4.2, 2.3, 3.2, 1.3, 4.3],
    [2.2, 3.7, 0.9, 3.8, 2.7, 2.8, 1.7, 4.7],
    [2.0, 3.5, 1.0, 4.0, 2.5, 3.0, 1.5, 4.5],
    [2.0, 3.5, 1.0, 4.0, 2.5, 3.0, 1.5, 4.5],
    [2.1, 3.4, 0.1, 4.0, 2.6, 1.1, 1.3, 2.5],
    [2.0, 3.5, 0.9, 4.1, 2.4, 3.0, 1.5, 4.4],
    [2.2, 3.3, 1.0, 4.2, 2.5, 3.0, 1.4, 4.6]
])

real_control.shape

(9, 8)

**Case Cells**

Next we'll show the case cell values. In our raw data, we have the real measured expression of the perturbed cell that's paired with a control cell of the same batch. Our simulated data will assume that our positions in the control and case match up based on batch and perturbation.

In [8]:
real_case = np.array([
    [2.8, 2.3, 1.6, 6.0, 2.0, 4.7, 1.2, 2.9],
    [2.8, 2.2, 1.0, 6.0, 2.0, 4.3, 1.6, 2.5],
    [2.1, 3.2, 1.2, 4.1, 2.4, 3.3, 1.3, 4.4],
    [2.3, 3.6, 0.9, 3.7, 2.7, 2.8, 1.7, 4.7],
    [2.5, 2.7, 1.6, 3.0, 2.8, 3.7, 1.1, 5.7],
    [2.3, 3.1, 1.2, 4.5, 2.2, 3.6, 1.4, 4.9],
    [2.8, 3.2, 1.8, 3.2, 2.4, 3.9, 1.1, 3.8],
    [4.4, 3.0, 1.7, 6.0, 0.2, 0.4, 1.1, 8.1],
    [1.6, 3.8, 8.5, 2.7, 6.8, 2.4, 1.8, 4.8]
])

real_case.shape

(9, 8)

**Calculate Sample Delta**

A major component of our perturbation retrieval benchmark is not looking at absolute predictions, but the change in expression (averaged by perturbation). Some claim that this simplifies the task. Biologically, we see this as addressing the important questions: can you predict what will change, in what direction, and by how much? For our perturbation retrieval, we focus on calculating a single sample-level difference we label as `real_delta`. This is the real change in expression as calculated by $\delta_g = x^{\text{case}}_g - x^{\text{ctrl}}_g$. This is our source of truth.

In [9]:
real_delta = real_case - real_control

real_delta.shape, real_delta

((9, 8),
 array([[ 0.7, -1.1,  0.4,  1.9, -0.6,  1.6, -0.2, -1.7],
        [ 0.9, -1.4,  0.2,  2.1, -0.4,  1.4,  0. , -1.9],
        [ 0.1, -0.1,  0.1, -0.1,  0.1,  0.1,  0. ,  0.1],
        [ 0.1, -0.1,  0. , -0.1,  0. ,  0. ,  0. ,  0. ],
        [ 0.5, -0.8,  0.6, -1. ,  0.3,  0.7, -0.4,  1.2],
        [ 0.3, -0.4,  0.2,  0.5, -0.3,  0.6, -0.1,  0.4],
        [ 0.7, -0.2,  1.7, -0.8, -0.2,  2.8, -0.2,  1.3],
        [ 2.4, -0.5,  0.8,  1.9, -2.2, -2.6, -0.4,  3.7],
        [-0.6,  0.5,  7.5, -1.5,  4.3, -0.6,  0.4,  0.2]]))

**Calculate Per-Perturbation Means**

Now we'll calculate our per-perturbation data. We'll use our `sample_to_pert` to help identify which perturbation each sample belongs to. We'll iterate through our perturbations, find which samples belong to the perturbation, pluck out the expression data for the samples, and then run a mean across them to get a single value per perturbation. This will become the basis of our perturbation-based retrieval.

In [10]:
mean_pert_control_abs = np.zeros((unique_perts, num_genes))
mean_pert_real_delta = np.zeros((unique_perts, num_genes))

In [11]:
for pert_idx in range(unique_perts):
    mask = [i for i, p in enumerate(sample_to_pert) if p == pert_idx]
    mean_pert_control_abs[pert_idx] = real_control[mask].mean(axis=0)
    mean_pert_real_delta[pert_idx] = real_delta[mask].mean(axis=0)

mean_pert_control_abs.shape, mean_pert_control_abs, mean_pert_real_delta

((7, 8),
 array([[2. , 3.5, 1. , 4. , 2.5, 3. , 1.5, 4.5],
        [2.1, 3.5, 1. , 4. , 2.5, 3. , 1.5, 4.5],
        [2. , 3.5, 1. , 4. , 2.5, 3. , 1.5, 4.5],
        [2. , 3.5, 1. , 4. , 2.5, 3. , 1.5, 4.5],
        [2.1, 3.4, 0.1, 4. , 2.6, 1.1, 1.3, 2.5],
        [2. , 3.5, 0.9, 4.1, 2.4, 3. , 1.5, 4.4],
        [2.2, 3.3, 1. , 4.2, 2.5, 3. , 1.4, 4.6]]),
 array([[ 0.8 , -1.25,  0.3 ,  2.  , -0.5 ,  1.5 , -0.1 , -1.8 ],
        [ 0.1 , -0.1 ,  0.05, -0.1 ,  0.05,  0.05,  0.  ,  0.05],
        [ 0.5 , -0.8 ,  0.6 , -1.  ,  0.3 ,  0.7 , -0.4 ,  1.2 ],
        [ 0.3 , -0.4 ,  0.2 ,  0.5 , -0.3 ,  0.6 , -0.1 ,  0.4 ],
        [ 0.7 , -0.2 ,  1.7 , -0.8 , -0.2 ,  2.8 , -0.2 ,  1.3 ],
        [ 2.4 , -0.5 ,  0.8 ,  1.9 , -2.2 , -2.6 , -0.4 ,  3.7 ],
        [-0.6 ,  0.5 ,  7.5 , -1.5 ,  4.3 , -0.6 ,  0.4 ,  0.2 ]]))

## Staged Inference Function

In our actual code, we run inference per perturbation key that we process. For our explainer, this would be tricky to set up. Instead we'll just create a function that returns the right staged information. As we loop through our perturbation retrieval analysis, you'll see how we use this data.

Since we've only staged a few perturbations per category, each returned row represents one candidate in our smaller candidate bank. In production, this same calculation runs across every entry in the corresponding embedding bank.

In [12]:
def get_inf(mean_control_states, key_bank_info):
    inf_dict = {
        0: np.array([
            [0.7, -1.1, 0.4, 1.8, -0.4, 1.3, -0.2, -1.6],
            [0.15, -0.05, -0.1, 0.05, 0.1, -0.05, 0.05, 0.0],
            [-0.3, 0.5, -0.4, 0.7, -0.1, -0.5, 0.2, -0.9],
        ]),
        1: np.array([
            [0.6, -0.9, 0.3, 1.5, -0.3, 1.1, -0.1, -1.4],
            [0.1, 0.05, -0.1, 0.05, 0.1, -0.05, 0.05, 0.0],
            [-0.2, 0.4, -0.3, 0.5, -0.1, -0.3, 0.1, -0.6],
        ]),
        2: np.array([
            [0.5, -0.8, 0.2, 1.4, -0.3, 1.0, -0.1, -1.2],
            [0.1, -0.05, -0.05, 0.0, 0.05, 0.0, 0.05, 0.0],
            [-0.4, 0.6, -0.5, 0.8, -0.2, -0.6, 0.3, -1.0],
        ]),
        3: np.array([
            [0.9, -1.2, 0.6, 1.5, -0.9, 1.8, -0.3, 1.2],
            [0.6, -0.1, 0.6, -0.7, -0.1, 0.7, -0.1, -0.6],
        ]),
        4: np.array([
            [0.2, -0.3, 0.1, 0.4, -0.2, 0.5, -0.1, 0.3],
            [0.5, -0.1, 0.5, -0.6, -0.1, 0.6, -0.1, -0.5],
        ]),
        5: np.array([
            [-0.3, 0.2, -0.3, -1.0, 0.3, -0.4, 0.2, 0.3],
            [0.3, -0.2, 0.6, 1.5, -0.2, 0.3, -0.2, -0.4],
        ]),
        6: np.array([
            [-0.3, 0.2, -0.3, -1.1, 0.3, -0.5, 0.2, 0.3],
            [0.2, -0.2, 0.5, 1.3, -0.2, 0.3, -0.2, -0.3],
        ])
    }

    return inf_dict[key_bank_info]

## DNA-Based Analysis

The first set of analyses will be done on our DNA-based perturbations. This is any perturbation where the sequence is DNA-based.

### Extract the Perturbations and Cell Expression Data

While our production eval loop will use the perturbation key (unique identifier of 5 traits) as the key to extract the different expression and perturbation data, we'll do a slightly different approach by indexing our data positionally using `modality_to_pertidx`. This positioning index will still allow us to demonstrate our calculations.

We'll extract both the perturbations and the expression data based on the index we have built.

In [13]:
ranks = []
eval_keys = modality_to_pertidx['dna']
eval_keys

[0, 1, 2]

### Per-Perturbation Analysis

Now that we know which perturbations are in our category, we actually need to loop through each one and run our similarity analysis. In this loop, we do the following:

1. Get the mean control expression for the query and the candidate bank that matches its retrieval type
2. Run inference applying every candidate in the bank to the query's control state
3. Calculate the cosine similarity
4. Calculate the rank

For DNA and chemical retrieval, the candidates use the sequence path through the composer. For target-only retrieval, the candidates use the target path. We also keep the query's mode fixed across every candidate so that we're comparing the perturbation input without changing the type of effect at the same time.

After we run this for all the perturbations, we calculate our retrieval metrics by category.

#### First Example Step by Step
Since this is a loop, I'll walk through the first example step by step, then write a loop to calculate the rest. We'll start with the first perturbation:

In [14]:
key = eval_keys[0]
key

0

#### Get Prediction of All Perturbations

Now we're ready to extract the expected impact of each candidate in the bank on the mean control for the indexed perturbation. This relies on taking the mean control absolute expression, encoding it through our teacher cell state encoder to get a latent representation, and then applying each candidate through the composer, predictor, and [linear expression decoder](explainer_eval_decoders_v1_0.ipynb) to generate a predicted expression delta. We calculate this as:

$$
\hat{\Delta}_c = \{\hat{\delta}_{p,c}\}_{p=0}^{N-1}, \quad \hat{\delta}_{p,c} = D(\hat{z}^{\text{pred}}_{p,c}) - D(\hat{z}^{\text{ctrl}}_{c})
$$

Where $c$ is the query (the test perturbation whose control state we're starting from), $p$ indexes each candidate perturbation in the bank, and $N$ is the total number of candidates. $D$ is the linear decoder, $\hat{z}^{\text{ctrl}}_{c}$ is the teacher encoding of query $c$'s mean control expression, and $\hat{z}^{\text{pred}}_{p,c}$ is the predictor output given that control latent plus candidate $p$'s action vector. The result $\hat{\Delta}_c$ is an $[N, G]$ matrix where each row is the predicted expression delta for one candidate perturbation applied to this query's control state.

*We have already staged our inference values in `get_inf` so, in our case, it'll just be a matter of calling that function to get our predicted deltas.*

In [15]:
mean_control_states = mean_pert_control_abs[key]
mean_control_states.shape, mean_control_states

((8,), array([2. , 3.5, 1. , 4. , 2.5, 3. , 1.5, 4.5]))

In [16]:
preds = get_inf(mean_control_states, key)
preds.shape, preds

((3, 8),
 array([[ 0.7 , -1.1 ,  0.4 ,  1.8 , -0.4 ,  1.3 , -0.2 , -1.6 ],
        [ 0.15, -0.05, -0.1 ,  0.05,  0.1 , -0.05,  0.05,  0.  ],
        [-0.3 ,  0.5 , -0.4 ,  0.7 , -0.1 , -0.5 ,  0.2 , -0.9 ]]))

#### Cosine Similarity

Now that we have the predicted expression changes produced by applying all candidates to our control, we're ready to analyze how close each one is to the real expression change. We do this using cosine similarity. Cosine similarity tells us how similar the direction of the predicted and real expression delta profiles are, ignoring magnitude. We calculate the similarity as:

$$
\text{cos\_sim}(\hat{\delta}_{p,c},\, \delta_c) = \frac{\hat{\delta}_{p,c} \cdot \delta_c}{\|\hat{\delta}_{p,c}\| \cdot \|\delta_c\|}
$$

Where $p$ indexes the candidate perturbation from the bank and $c$ is the query (test perturbation). The higher the value, the more directionally aligned the predicted and real expression delta profiles are. You'll see that, because we staged the data to have a great prediction with our first perturbation, the similarity is best on that index.

In [17]:
mean_real_deltas = mean_pert_real_delta[key]
mean_real_deltas.shape, mean_real_deltas

((8,), array([ 0.8 , -1.25,  0.3 ,  2.  , -0.5 ,  1.5 , -0.1 , -1.8 ]))

In [18]:
pred_n = preds / (np.linalg.norm(preds, axis=-1, keepdims=True) + 1e-8)
pred_n.shape, pred_n

((3, 8),
 array([[ 0.22651468, -0.35595164,  0.12943696,  0.58246632, -0.12943696,
          0.42067012, -0.06471848, -0.51774784],
        [ 0.65465364, -0.21821788, -0.43643576,  0.21821788,  0.43643576,
         -0.21821788,  0.21821788,  0.        ],
        [-0.20701967,  0.34503278, -0.27602622,  0.48304589, -0.06900656,
         -0.34503278,  0.13801311, -0.621059  ]]))

In [19]:
real_n = mean_real_deltas / (np.linalg.norm(mean_real_deltas) + 1e-8)
real_n.shape, real_n

((8,),
 array([ 0.23053223, -0.36020662,  0.08644959,  0.57633058, -0.14408265,
         0.43224794, -0.02881653, -0.51869753]))

In [20]:
sims = np.dot(pred_n, real_n)
sims

array([0.99822089, 0.15406291, 0.26149162])

#### Rank

Now that we have the similarity comparison between the real expression delta and each candidate's predicted impact on the control, we need to identify where the true perturbation lands. We rank the candidates by their cosine similarity to the real delta:

$$
\text{rank}_{c} = \text{position of } p_c \text{ in } \text{argsort}(\text{cos\_sim}(\hat{\Delta}_c, \delta_c))_{\text{desc}} + 1
$$

Where $p_c$ is the true perturbation's index in the bank (the lookup_idx). We sort all candidates by descending cosine similarity and find where the true perturbation lands. The +1 converts from 0-indexed to 1-indexed, so rank 1 means the true perturbation had the highest similarity. When you have a high number of genes that don't move for a given perturbation, the cosine similarity is dominated by noise in the near-zero genes rather than the signal in the DEGs. (A future improvement could be using a top DEG filter to remove the noise.)

You'll see how the similarity and rank change based on the perturbation and our staged data. For this first example, since our model was great at predicting, we'll see it at rank 1.

In [21]:
rank = int(np.where(np.argsort(sims)[::-1] == key)[0][0]) + 1
ranks.append(rank)
rank, ranks

(1, [1])

#### Loop Through Remaining Perturbations

Now that we've walked through how we calculate similarity for a single perturbation, we'll loop through the remaining DNA perturbations.

*You'll see that our loop needs an extra +1 to skip the first position.*

In [22]:
for k in range(len(eval_keys[1:])):
    key = eval_keys[k + 1]
    print(f'---- Evaluating Pert {key} ----')
    mean_control_states = mean_pert_control_abs[key]
    preds = get_inf(mean_control_states, key)
    mean_real_deltas = mean_pert_real_delta[key]

    print(preds)
    print(mean_real_deltas)
    # cos sim
    pred_n = preds / (np.linalg.norm(preds, axis=-1, keepdims=True) + 1e-8)
    real_n = mean_real_deltas / (np.linalg.norm(mean_real_deltas) + 1e-8)
    sims = np.dot(pred_n, real_n)

    # rank
    rank = int(np.where(np.argsort(sims)[::-1] == k + 1)[0][0]) + 1
    ranks.append(rank)

    print(f'Perturbation similarity: sim {sims} | rank {rank}')

---- Evaluating Pert 1 ----
[[ 0.6  -0.9   0.3   1.5  -0.3   1.1  -0.1  -1.4 ]
 [ 0.1   0.05 -0.1   0.05  0.1  -0.05  0.05  0.  ]
 [-0.2   0.4  -0.3   0.5  -0.1  -0.3   0.1  -0.6 ]]
[ 0.1  -0.1   0.05 -0.1   0.05  0.05  0.    0.05]
Perturbation similarity: sim [-0.02880357 -0.06249999 -0.87065749] | rank 2
---- Evaluating Pert 2 ----
[[ 0.5  -0.8   0.2   1.4  -0.3   1.   -0.1  -1.2 ]
 [ 0.1  -0.05 -0.05  0.    0.05  0.    0.05  0.  ]
 [-0.4   0.6  -0.5   0.8  -0.2  -0.6   0.3  -1.  ]]
[ 0.5 -0.8  0.6 -1.   0.3  0.7 -0.4  1.2]
Perturbation similarity: sim [-0.24059147  0.1847761  -0.99880834] | rank 3


**Ranks**

Now that we've calculated all the similarities and ranks, we can see the diversity in our results. We've purposefully staged our data to show how different predicted profiles produce different similarities and ranks.

In [23]:
ranks = np.array(ranks)
ranks

array([1, 2, 3])

#### Mean Reciprocal Rank (MRR)

Our first metric measures how quickly we find the correct perturbation on average. Across the perturbations in this retrieval type, it tells us how accurately the model retrieves the right candidate. We calculate MRR as:

$$
\text{MRR} = \frac{1}{|\mathcal{E}|} \sum_{c \in \mathcal{E}} \frac{1}{\text{rank}_c}
$$

Where $\mathcal{E}$ is the set of evaluated perturbations and $\text{rank}_c$ is the position of the true perturbation in the similarity-sorted candidate list. A perfect retriever scores 1.0 (every true perturbation ranked first). MRR penalizes lower ranks heavily: rank 1 contributes 1.0, rank 2 contributes 0.5, rank 10 contributes only 0.1. This makes it sensitive to whether the correct answer appears near the top of the list. With our ranks of $[1, 2, 3]$, we first convert them to $[1, \frac{1}{2}, \frac{1}{3}]$ and then take the mean, giving us an MRR of approximately $0.611$. This shows how the first positions matter more: moving from rank 1 to rank 2 cuts that query's contribution in half.

In [24]:
dna_mrr = float(np.mean(1.0 / ranks))
dna_mrr

0.611111111111111

#### Median Rank

Median rank is the middle value when all ranks are sorted. This is unweighted and gives us a sense of how many candidates typically rank ahead of the true perturbation. We calculate it as:

$$
\text{median\_rank} = \text{median}(\{\text{rank}_c\}_{c \in \mathcal{E}})
$$

With median rank, lower is better. Compared to mean rank, the median is less sensitive to outliers, so a single poorly ranked perturbation won't skew the result. A median rank of 1 means at least half the perturbations were correctly identified as the top candidate. You'll see that with 3 predictions, we'll get $2.0$ as the value.

In [25]:
dna_median = float(np.median(ranks))
dna_median

2.0

#### Recall @ K

Recall @ K measures what fraction of perturbations have their true match appear in the top $K$ candidates. This analysis is helpful if we turned this eval into a prediction tool. We calculate recall as:

$$
\text{Recall@K} = \frac{1}{|\mathcal{E}|} \sum_{c \in \mathcal{E}} \mathbb{1}[\text{rank}_c \leq K]
$$

Where $\mathbb{1}[\cdot]$ is the indicator function. We evaluate at multiple $K$ values to show how quickly recall improves as we allow more candidates. Recall @ 1 asks "how often is the correct perturbation ranked first?" while Recall @ 3 asks "how often is it in the top 3?" In our staged data, the DNA candidate bank has 3 entries, so Recall @ 3 reaches $1.0$. In production, $K$ is limited by the size of the candidate bank, not the number of queries we evaluate. We report Recall @ 1, 5, 10, 20, and 50 when the bank is large enough.

In [26]:
K_VALUES = [1, 2, 3]
dna_recall_k = {str(k): float(np.mean(ranks <= k)) for k in K_VALUES if k <= len(eval_keys)}
dna_recall_k

{'1': 0.3333333333333333, '2': 0.6666666666666666, '3': 1.0}

## Chemical-Based Analysis

The next set of analyses will be based on the chemical perturbations. This is any perturbation where the sequence is SMILES-based.

*Note that we'll only comment where the code changes or if we think extra commentary is needed*

### Extract the Perturbations and Cell Expression Data

For this loop, we'll extract the chemical-based perturbations. You'll see that we get the two that we staged.

In [27]:
ranks = []
eval_keys = modality_to_pertidx['chemical']
eval_keys

[3, 4]

### Per-Perturbation Analysis

We'll run the same similarity loop on our chemical perturbations. You'll see that this time we can run a clean loop without any extra $+1$ added. A key difference for the chemical perturbations is that we only have 2 candidates. Because we have fewer candidates, our ranks and retrieval metrics should improve.

We've staged the first example to rank correctly, but our second example, while very close, will not be the best fit.

#### Loop Through Evals

In [28]:
ranks = []

In [29]:
for k in range(len(eval_keys)):
    key = eval_keys[k]
    print(f'---- Evaluating Pert {key} ----')
    mean_control_states = mean_pert_control_abs[key]
    preds = get_inf(mean_control_states, key)
    mean_real_deltas = mean_pert_real_delta[key]

    print(preds)
    print(mean_real_deltas)
    # cos sim
    pred_n = preds / (np.linalg.norm(preds, axis=-1, keepdims=True) + 1e-8)
    real_n = mean_real_deltas / (np.linalg.norm(mean_real_deltas) + 1e-8)
    sims = np.dot(pred_n, real_n)
    print(sims)
    # rank
    rank = int(np.where(np.argsort(sims)[::-1] == k)[0][0]) + 1
    ranks.append(rank)

    print(f'Perturbation similarity: sim {sims} | rank {rank}')

---- Evaluating Pert 3 ----
[[ 0.9 -1.2  0.6  1.5 -0.9  1.8 -0.3  1.2]
 [ 0.6 -0.1  0.6 -0.7 -0.1  0.7 -0.1 -0.6]]
[ 0.3 -0.4  0.2  0.5 -0.3  0.6 -0.1  0.4]
[0.99999999 0.13487056]
Perturbation similarity: sim [0.99999999 0.13487056] | rank 1
---- Evaluating Pert 4 ----
[[ 0.2 -0.3  0.1  0.4 -0.2  0.5 -0.1  0.3]
 [ 0.5 -0.1  0.5 -0.6 -0.1  0.6 -0.1 -0.5]]
[ 0.7 -0.2  1.7 -0.8 -0.2  2.8 -0.2  1.3]
[0.61864968 0.61171605]
Perturbation similarity: sim [0.61864968 0.61171605] | rank 2


**Ranks**

In [30]:
ranks = np.array(ranks)
ranks

array([1, 2])

#### MRR

In [31]:
chem_mrr = float(np.mean(1.0 / ranks))
chem_mrr

0.75

#### Median Rank

In [32]:
chem_median = float(np.median(ranks))
chem_median

1.5

#### Recall @ K

In [33]:
K_VALUES = [1, 2, 3]
chem_recall_k = {str(k): float(np.mean(ranks <= k)) for k in K_VALUES if k <= len(eval_keys)}
chem_recall_k

{'1': 0.5, '2': 1.0}

## Target-Only Perturbation Analysis

The next set of analyses will be based on the perturbations that only have a target and no sequence. This can be a DNA- or chemical-based perturbation that's just missing the sequence in our training data.

*Note that we'll only comment where the code changes or if we think extra commentary is needed*

### Extract the Perturbations and Cell Expression Data

We'll extract the perturbations for target-only. This will be the last two perturbations in our dataset.

In [34]:
ranks = []
eval_keys = modality_to_pertidx['target_only']
eval_keys

[5, 6]

### Per-Perturbation Analysis

We'll run the same similarity evaluation with the target-only perturbations. For this set of perturbations, we've created a particularly bad predicted example to show that our cosine similarity can actually go negative. We've also staged both examples to rank second to show how the metrics change when there are no rank-1 examples.

#### Loop Through Evals

In [35]:
ranks = []

In [36]:
for k in range(len(eval_keys)):
    key = eval_keys[k]
    print(f'---- Evaluating Pert {key} ----')
    mean_control_states = mean_pert_control_abs[key]
    preds = get_inf(mean_control_states, key)
    mean_real_deltas = mean_pert_real_delta[key]

    print(preds)
    print(mean_real_deltas)
    # cos sim
    pred_n = preds / (np.linalg.norm(preds, axis=-1, keepdims=True) + 1e-8)
    real_n = mean_real_deltas / (np.linalg.norm(mean_real_deltas) + 1e-8)
    sims = np.dot(pred_n, real_n)
    print(sims)
    # rank
    rank = int(np.where(np.argsort(sims)[::-1] == k)[0][0]) + 1
    ranks.append(rank)

    print(f'Perturbation similarity: sim {sims} | rank {rank}')

---- Evaluating Pert 5 ----
[[-0.3  0.2 -0.3 -1.   0.3 -0.4  0.2  0.3]
 [ 0.3 -0.2  0.6  1.5 -0.2  0.3 -0.2 -0.4]]
[ 2.4 -0.5  0.8  1.9 -2.2 -2.6 -0.4  3.7]
[-0.20505803  0.23017223]
Perturbation similarity: sim [-0.20505803  0.23017223] | rank 2
---- Evaluating Pert 6 ----
[[-0.3  0.2 -0.3 -1.1  0.3 -0.5  0.2  0.3]
 [ 0.2 -0.2  0.5  1.3 -0.2  0.3 -0.2 -0.3]]
[-0.6  0.5  7.5 -1.5  4.3 -0.6  0.4  0.2]
[0.11570444 0.02996403]
Perturbation similarity: sim [0.11570444 0.02996403] | rank 2


**Ranks**

Now that we've calculated all the similarities and ranks, we can see the diversity in our results. For the first time across the categories, neither true perturbation ranks first. You'll see how this propagates to the remaining calculations.

In [37]:
ranks = np.array(ranks)
ranks

array([2, 2])

#### MRR

In [38]:
targ_mrr = float(np.mean(1.0 / ranks))
targ_mrr

0.5

#### Median Rank

In [39]:
targ_median = float(np.median(ranks))
targ_median

2.0

#### Recall @ K

In [40]:
K_VALUES = [1, 2, 3]
targ_recall_k = {str(k): float(np.mean(ranks <= k)) for k in K_VALUES if k <= len(eval_keys)}
targ_recall_k

{'1': 0.0, '2': 1.0}

## Summary

We've walked through the perturbation retrieval evaluation for all three modality categories. This evaluation moves beyond just "expression prediction" metrics by exploring different use cases of the foundation model. By reviewing how well our model predicts the actual perturbation profile against other perturbations, we can start to see how the model could be flipped to derive a perturbation from a cell's state compared to its control. We've also shown with our staged examples the downside of this evaluation focusing on cosine similarity, which ignores magnitude. Overall, this evaluation shows how the BioJEPA-AC model goes beyond simple expression prediction and can be used for other tasks as well.